In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import mhnlib.utils as mhn_utils
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from math import sqrt, log

In [ ]:
from torchvision.datasets import CIFAR10, CIFAR100, MNIST, STL10
import torchvision.transforms as T

In [ ]:
DATASET = "cifar100"
IMG_SIZE = 64
tfm = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.Grayscale(num_output_channels=3),
    T.ToTensor(),
])
if DATASET == "cifar100":
    dataset = CIFAR100(root="datasets/cifar100/", train=True, download=True, transform=tfm)
    test_set = CIFAR100(root="datasets/cifar100/", train=False, download=True, transform=tfm)
elif DATASET == "cifar10":
    dataset = CIFAR10(root="datasets/cifar10/", train=True, download=True, transform=tfm)
    test_set = CIFAR10(root="datasets/cifar10/", train=False, download=True, transform=tfm)
elif DATASET == "stl10":
    dataset = STL10(root="datasets/stl10/", split="train", download=True, transform=tfm)
    test_set = STL10(root="datasets/stl10/", split="test", download=True, transform=tfm)
elif DATASET == "mnist":
    dataset = MNIST(root="datasets/mnist/", train=True, download=True, transform=tfm)
    test_set = MNIST(root="datasets/mnist/", train=False, download=True, transform=tfm)

In [ ]:
from diffusers import AutoencoderKL
ae_model = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
ae_model = ae_model.to(device).eval()
ae_model.requires_grad_(False)
#ae_scaling = ae_model.config.scaling_factor

In [ ]:
num_memories = 256

estimation_dataloader = DataLoader(
    dataset,
    batch_size=num_memories,
    shuffle=True,
    num_workers=4
)

num_batches_for_estimation = 100

sum_z = None
sum_sq_norm = 0.0
n_samples = 0

# ------------------------------------------------------------
# 1. Estimate global latent mean and global squared norm
# ------------------------------------------------------------

with torch.no_grad():
    for i, (x, _) in tqdm(enumerate(estimation_dataloader)):
        if i >= num_batches_for_estimation:
            break

        x = x.to(device)

        z = ae_model.encode(x).latent_dist.mode().cpu()

        C_latents, H_latents, W_latents = z.shape[1:]

        if sum_z is None:
            sum_z = torch.zeros_like(z[0])

        sum_z += z.sum(dim=0)
        sum_sq_norm += z.pow(2).sum().item()
        n_samples += z.shape[0]

mean = sum_z / n_samples

mean_squared_norm = (
    sum_sq_norm / n_samples
    - mean.pow(2).sum().item()
)

scale = mean_squared_norm**0.5

# ------------------------------------------------------------
# 2. Estimate beta_c using the SAME global normalization
# ------------------------------------------------------------

all_beta_c = []

with torch.no_grad():
    for i, (x, _) in tqdm(enumerate(estimation_dataloader)):
        if i >= num_batches_for_estimation:
            break

        x = x.to(device)

        z = ae_model.encode(x).latent_dist.mode().cpu()

        z_transf = (z - mean) / scale
        z_transf_flattened = z_transf.flatten(1)

        gram = z_transf_flattened @ z_transf_flattened.T

        w = torch.ones(z.size(0)) / z.size(0)

        stability_matrix = (
            mhn_utils.get_dual_symmetric_stability_matrix(
                gram,
                w
            )
        )

        lambda_max = torch.linalg.eigvalsh(
            stability_matrix
        )[-1].item()

        beta_c = 1.0 / lambda_max
        all_beta_c.append(beta_c)

all_beta_c = torch.tensor(all_beta_c)

# Robust estimate of the typical critical beta
beta_c = all_beta_c.median().item()

In [ ]:
# ------------------------------------------------------------
# 3. Center noise sampling around beta_c
#
# beta = 1 / sigma^2
# => log sigma_c = -1/2 log beta_c
# ------------------------------------------------------------

log_noise_mean = -0.5 * log(beta_c)

# Width in log(sigma).
# Note: std(log beta) = 2 * log_noise_std
log_noise_std = 1.0

print(f"mean squared latent norm: {mean_squared_norm:.4f}")
print(f"scale:                    {scale:.4f}")
print(f"median beta_c:            {beta_c:.4f}")
print(f"log_noise_mean:           {log_noise_mean:.4f}")
print(f"log_noise_std:            {log_noise_std:.4f}")

In [ ]:
latent_dim = C_latents * H_latents * W_latents

memories = torch.randn(
    num_memories,
    C_latents,
    H_latents,
    W_latents,
    device=device,
) / sqrt(latent_dim)

memories.requires_grad_(True)

dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
)

num_epochs = 50
optimizer = optim.Adam([memories], lr=1e-3)

mean_device = mean.to(device)

pbar = tqdm(range(num_epochs), desc="Training")

for epoch in pbar:
    for i, (x, _) in enumerate(dataloader):

        pbar.set_description(
            f"Training epoch {epoch+1}/{num_epochs}, "
            f"batch {i+1}/{len(dataloader)}"
        )

        with torch.no_grad():
            x = x.to(device)

            # Use the same latent representation as during estimation
            z = ae_model.encode(x).latent_dist.mode()

            # Global centering + global norm rescaling
            z_transf = (z - mean_device) / scale

        B = z_transf.shape[0]

        # Sample one noise level per datapoint
        #noise_level = torch.exp(
        #    log_noise_mean
        #    + log_noise_std * torch.randn(B, device=device)
        #)
        #
        #beta = noise_level.pow(-2)
        beta_seed = torch.rand(B, device=device)
        beta = beta_seed/(1-beta_seed)
        noise_level = beta.rsqrt()

        # y = z_0 + sigma * epsilon
        z_transf_noised = (
            z_transf
            + noise_level[:, None, None, None]
            * torch.randn_like(z_transf)
        )

        # ||xi_mu||^2
        memories_sq_norms = memories.pow(2).sum(
            dim=(1, 2, 3)
        )

        # y . xi_mu
        overlaps = torch.einsum(
            "kchw,bchw->bk",
            memories,
            z_transf_noised,
        )

        # lambda * (y . xi_mu - ||xi_mu||^2 / 2)
        logits = beta[:, None] * (
            overlaps
            - 0.5 * memories_sq_norms[None, :]
        )

        weights = F.softmax(logits, dim=1)

        # m_lambda(y)
        predicted_z_transf = torch.einsum(
            "bk,kchw->bchw",
            weights,
            memories,
        )

        # Posterior-mean regression loss
        loss = F.mse_loss(
            predicted_z_transf,
            z_transf,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pbar.set_postfix(loss=loss.item())

In [ ]:
with torch.no_grad():
    memories_learned = memories.detach().cpu()
    # Undo the global latent normalization
    latent_space_memories = (
        memories.detach().cpu() * scale + mean
    )

    # Decode in the original AE latent space
    latent_space_memories_device = latent_space_memories.to(
        device=device,
        dtype=next(ae_model.parameters()).dtype
    )

    true_space_memories = (
        ae_model.decode(latent_space_memories_device)
        .sample
        .cpu()
    )

In [ ]:
for i in range(num_memories):
    plt.imshow( true_space_memories[i].mean(dim=0))
    plt.axis('off')
    plt.title(f'Memory {i+1}')
    plt.show()